# Simulação de confrontos diretos e de temporada regular
* Serão capturados os últimos elos registrados para cada time;
* Será testado o resultado de um confronto direto;
* Será simulado um torneio sem monte carlo, e com monte carlo.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

In [3]:
ROOT = Path.cwd().parent
df_elo = pd.read_csv(ROOT /'historico_nbb' / 'data' / 'historico_elo.csv')
nbb_26 = pd.read_csv(ROOT /'historico_nbb' / 'data' / 'calendario_26_nbb.csv')

In [4]:
df_elo.head()

,Season,Time,Elo
0,2016,Bauru,1488.577302
1,2016,Flamengo,1511.422698
2,2016,Pinheiros,1488.295954
3,2016,Vitória,1511.704046
4,2016,Minas,1507.678613


In [5]:
nbb_26.head()

,Season,Casa,Visitante
0,2026,Mogi,Corinthians
1,2026,Caxias do Sul Basquete,Cruzeiro
2,2026,UNIFACISA,Basket Osasco
3,2026,Flamengo,Bauru
4,2026,Franca,São José


In [ ]:
df_elo['Time'].unique() # Vendo a lista dos times que estão no histórico de elo

<StringArray>
[                 'Bauru',               'Flamengo',              'Pinheiros',
                'Vitória',                  'Minas', 'Caxias do Sul Basquete',
                 'Franca',         'Macaé Basquete',      'Basquete Cearense',
          'Vasco da Gama',               'Brasília',           'Campo Mourão',
          'L. Sorocabana',                   'Mogi',             'Paulistano',
       'Joinville / AABJ',               'Botafogo',               'São José',
            'Corinthians',            'VipTech CMB',              'UNIFACISA',
          'Pato Basquete',              'São Paulo',       'Cerrado Basquete',
      'União Corinthians',              'Rio Claro',               'Cruzeiro',
          'Basket Osasco']
Length: 28, dtype: str

In [ ]:
nbb_26['Casa'].unique() # Vendo a lista dos times que estão no calendário da temporada 2026

<StringArray>
[                  'Mogi', 'Caxias do Sul Basquete',              'UNIFACISA',
               'Flamengo',                 'Franca',      'União Corinthians',
      'Basquete Cearense',                  'Bauru',          'Pato Basquete',
               'Brasília',            'Corinthians',                  'Minas',
               'Cruzeiro',              'Pinheiros',             'Paulistano',
               'São José',               'Botafogo',          'Basket Osasco']
Length: 18, dtype: str

In [10]:
# ===========================================================
# Coletando o último elo de cada time para a temporada 2026
# ===========================================================
elo_26 = df_elo[df_elo['Season'] == 2025].groupby('Time').last().reset_index()[['Time', 'Elo']]

elo_26

,Time,Elo
0,Basket Osasco,1282.563945
1,Basquete Cearense,1244.923036
2,Bauru,1594.751875
3,Botafogo,1352.330969
4,Brasília,1762.364740
5,Caxias do Sul Basquete,1492.955483
6,Corinthians,1771.403159
7,Cruzeiro,1442.496105
8,Flamengo,1714.594660
9,Franca,1919.996108


In [11]:
# ===========================================================
# Simulando uma partida entre equipes selecionadas
# ===========================================================
bsb = elo_26[elo_26['Time'] == 'Brasília'].iloc[0]['Elo']
fla = elo_26[elo_26['Time'] == 'Flamengo'].iloc[0]['Elo']

prob_bsb = 1 / (1 + 10 ** ((fla - bsb) / 400))

result = np.random.choice(['Brasília', 'Flamengo'], p=[prob_bsb, 1 - prob_bsb])

print(f"Probabilidade de vitória do Brasília: {prob_bsb:.2%}")
print(f"Probabilidade de vitória do Flamengo: {1 - prob_bsb:.2%}")
print(f"Resultado da partida: {result}")

Probabilidade de vitória do Brasília: 56.83%
Probabilidade de vitória do Flamengo: 43.17%
Resultado da partida: Brasília


In [13]:
# ====================================================================================
# Simulando uma partida entre equipes selecionadas com vantagem de casa
# ====================================================================================
casa = 60
# Supondo que o Brasília jogue em casa
bsb = elo_26[elo_26['Time'] == 'Brasília'].iloc[0]['Elo']
fla = elo_26[elo_26['Time'] == 'Flamengo'].iloc[0]['Elo']
vtg =  (bsb + casa) - fla

prob_bsb = 1 / (1 + 10 ** ((-vtg) / 400))

result = np.random.choice(['Brasília', 'Flamengo'], p=[prob_bsb, 1 - prob_bsb])

print(f"Probabilidade de vitória do Brasília: {prob_bsb:.2%}")
print(f"Probabilidade de vitória do Flamengo: {1 - prob_bsb:.2%}")
print(f"Resultado da partida: {result}")

Probabilidade de vitória do Brasília: 65.03%
Probabilidade de vitória do Flamengo: 34.97%
Resultado da partida: Flamengo


In [14]:
# ====================================================================================
# Simulando uma partida entre equipes selecionadas com vantagem de casa várias vezes
# ====================================================================================
casa = 60
# Supondo que o Brasília jogue em casa
bsb = elo_26[elo_26['Time'] == 'Brasília'].iloc[0]['Elo']
fla = elo_26[elo_26['Time'] == 'Flamengo'].iloc[0]['Elo']
vtg =  (bsb + casa) - fla

prob_bsb = 1 / (1 + 10 ** ((-vtg) / 400))

result = []
for _ in range(5):  # Simula 5 partidas
    result.append(np.random.choice(['Brasília', 'Flamengo'], p=[prob_bsb, 1 - prob_bsb]))


print(f"Resultado da partida: {result}")

Resultado da partida: [np.str_('Brasília'), np.str_('Flamengo'), np.str_('Flamengo'), np.str_('Brasília'), np.str_('Flamengo')]


In [16]:
# ====================================================================================
# Simulando uma temporada da NBB 2026 com base no histórico de elo
# ====================================================================================
for row in nbb_26.itertuples():
    casa = row.Casa
    fora = row.Visitante
    
    elo_casa = elo_26[elo_26['Time'] == casa].iloc[0]['Elo']
    elo_fora = elo_26[elo_26['Time'] == fora].iloc[0]['Elo']
    vtg = (elo_casa + 60) - elo_fora  # Adicionando vantagem de casa de 60 pontos
    
    prob_casa = 1 / (1 + 10 ** ((-vtg) / 400))
    
    result = np.random.choice([casa, fora], p=[prob_casa, 1 - prob_casa])
    
    print(f"Jogo: {casa} vs {fora} | Resultado: {result} | Probabilidade de vitória do {casa}: {prob_casa:.2%}")

Jogo: Mogi vs Corinthians | Resultado: Corinthians | Probabilidade de vitória do Mogi: 48.41%
Jogo: Caxias do Sul Basquete vs Cruzeiro | Resultado: Cruzeiro | Probabilidade de vitória do Caxias do Sul Basquete: 65.38%
Jogo: UNIFACISA vs Basket Osasco | Resultado: UNIFACISA | Probabilidade de vitória do UNIFACISA: 89.31%
Jogo: Flamengo vs Bauru | Resultado: Flamengo | Probabilidade de vitória do Flamengo: 73.79%
Jogo: Franca vs São José | Resultado: Franca | Probabilidade de vitória do Franca: 92.77%
Jogo: União Corinthians vs Cruzeiro | Resultado: União Corinthians | Probabilidade de vitória do União Corinthians: 67.91%
Jogo: UNIFACISA vs Pinheiros | Resultado: Pinheiros | Probabilidade de vitória do UNIFACISA: 25.33%
Jogo: Caxias do Sul Basquete vs Minas | Resultado: Minas | Probabilidade de vitória do Caxias do Sul Basquete: 28.55%
Jogo: Basquete Cearense vs Basket Osasco | Resultado: Basquete Cearense | Probabilidade de vitória do Basquete Cearense: 53.21%
Jogo: Flamengo vs Botafogo